# אימון (Fine-tuning) של HeBERT לזיהוי מצוקה
פרויקט SafeSignal — סיווג בינארי: `0` = ניטרלי, `1` = מצוקה.

**לפני שמתחילים:** `Runtime -> Change runtime type -> T4 GPU` (חשוב! אחרת האימון יהיה איטי מאוד).

## 1. התקנת חבילות

In [2]:
!pip install -q transformers datasets accelerate scikit-learn evaluate


In [3]:
!pip uninstall -y torchvision


Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


## 2. העלאת קובץ הדאטא
לחצו על התא, הריצו, ואז בחרו את `distress_dataset.csv` מהמחשב שלכם.

In [4]:
from google.colab import files
uploaded = files.upload()  # בחרו את distress_dataset.csv


Saving distress_dataset.csv to distress_dataset.csv


## 3. טעינת הדאטא ובדיקה

In [5]:
import pandas as pd

df = pd.read_csv("distress_dataset.csv", encoding="utf-8-sig")
df = df[["text", "label"]].dropna()
df["label"] = df["label"].astype(int)

print("סה\"כ שורות:", len(df))
print(df["label"].value_counts())
df.head()


סה"כ שורות: 5736
label
1    3637
0    2099
Name: count, dtype: int64


,text,label
0,"kinda, אני לגמרי גמור אחרי האימון הזה hahaha!!",0
1,"not gonna lie, אני מפחד לצאת מהבית היום",1
2,יש לי כאלה הרבה מכרים אבל אף חבר אמיתי אחד??,1
3,"vibes, אני מת על הפרק האחרון, כתבו את זה מושלם",0
4,"אי אפשר להירגע, המחשבות פשוט לאנעצרות. אני מגי...",1


## 4. חלוקה ל-Train / Validation / Test (80/10/10, stratified)
ה-stratify שומר על אותו יחס 0/1 בכל חלק.

In [6]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

print("Train:", len(train_df), dict(train_df["label"].value_counts()))
print("Val:  ", len(val_df), dict(val_df["label"].value_counts()))
print("Test: ", len(test_df), dict(test_df["label"].value_counts()))


Train: 4588 {1: np.int64(2909), 0: np.int64(1679)}
Val:   574 {1: np.int64(364), 0: np.int64(210)}
Test:  574 {1: np.int64(364), 0: np.int64(210)}


## 5. טוקניזציה

In [7]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "avichr/heBERT"  # מודל הבסיס (לא הראש של סנטימנט) - נאמן ראש סיווג חדש לבינארי
MAX_LENGTH = 64  # המשפטים בדאטא קצרים (עד ~28 מילים), 64 טוקנים מספיק בנוחות

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(d):
    return Dataset.from_pandas(d[["text", "label"]].reset_index(drop=True))

train_ds = to_dataset(train_df)
val_ds = to_dataset(val_df)
test_ds = to_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

columns = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=columns)
val_ds.set_format(type="torch", columns=columns)
test_ds.set_format(type="torch", columns=columns)


config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/299k [00:00<?, ?B/s]

Map:   0%|          | 0/4588 [00:00<?, ? examples/s]

Map:   0%|          | 0/574 [00:00<?, ? examples/s]

Map:   0%|          | 0/574 [00:00<?, ? examples/s]

## 6. משקלי מחלקה (Class Weights)
הדאטא לא מאוזן (בערך 30% אפס / 70% אחד). במקום למחוק/לשכפל דוגמאות, נותנים למחלקה הקטנה יותר משקל גבוה יותר ב-loss.

In [8]:
import torch
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights [0, 1]:", class_weights)


Class weights [0, 1]: tensor([1.3663, 0.7886])


## 7. המודל וה-Trainer (עם weighted loss)

In [9]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "NEUTRAL", 1: "DISTRESS"},
    label2id={"NEUTRAL": 0, "DISTRESS": 1},
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


Device: cuda


pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: avichr/heBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly 

## 8. מדדי הערכה (Precision / Recall / F1 לכל מחלקה)
ב-screening למצוקה, recall על מחלקה 1 הוא הכי קריטי (פחות פספוסים).

In [10]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    p1, r1, f1_1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[1])
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "f1_macro": f1,
        "precision_macro": precision,
        "recall_macro": recall,
        "recall_distress": r1[0],
        "precision_distress": p1[0],
    }


## 9. הגדרת האימון והרצה

In [11]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="recall_distress",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro,Recall Distress,Precision Distress
1,0.053369,0.025487,0.993031,0.992506,0.991543,0.993498,0.991758,0.997238
2,0.000182,0.022107,0.994774,0.994374,0.993883,0.994872,0.994505,0.997245
3,0.000072,0.020358,0.994774,0.994374,0.993883,0.994872,0.994505,0.997245
4,0.000063,0.024691,0.994774,0.994374,0.993883,0.994872,0.994505,0.997245


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1148, training_loss=0.02727985688714791, metrics={'train_runtime': 249.61, 'train_samples_per_second': 73.523, 'train_steps_per_second': 4.599, 'total_flos': 603576760995840.0, 'train_loss': 0.02727985688714791, 'epoch': 4.0})

## 10. הערכה סופית על ה-Test Set (לא נגעו בו עד עכשיו)

In [12]:
from sklearn.metrics import classification_report, confusion_matrix

test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=1)
test_labels = test_output.label_ids

print(classification_report(test_labels, test_preds, target_names=["NEUTRAL(0)", "DISTRESS(1)"]))
print("Confusion matrix:")
print(confusion_matrix(test_labels, test_preds))


              precision    recall  f1-score   support

  NEUTRAL(0)       1.00      0.99      0.99       210
 DISTRESS(1)       0.99      1.00      1.00       364

    accuracy                           0.99       574
   macro avg       1.00      0.99      0.99       574
weighted avg       0.99      0.99      0.99       574

Confusion matrix:
[[207   3]
 [  0 364]]


## 10.5. בדיקת שפיות על משפטים חדשים (לא מהדאטאסט)
המטרה: לוודא שהמודל לא רק שינן תבניות מהדאטאסט, אלא באמת מזהה מצוקה על ניסוחים חדשים שלא ראה.

In [13]:
test_sentences = [
    "כל הכיתה פתחו עליי קבוצת מימים",
    "לא ישנתי שנתיים",
    "השארתי לכם מכתב",
    "מה אם לא תזרח השמש מחר אולי יהיה יום אחר",
    "אני לא יודע מה לעשות",
    "קניתי מחשב ואני מתלבט את מי להזמין לשחק אוף",
    "אף פעם אל תאמר אף פעם",
]

model.eval()
model.to(device)

inputs = tokenizer(test_sentences, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LENGTH).to(device)
with torch.no_grad():
    logits = model(**inputs).logits
probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)

for sent, pred, prob in zip(test_sentences, preds, probs):
    label = "DISTRESS (1)" if pred.item() == 1 else "NEUTRAL (0)"
    confidence = prob[pred.item()].item()
    print(f"[{label}]  (confidence={confidence:.3f})  {sent}")


[DISTRESS (1)]  (confidence=1.000)  כל הכיתה פתחו עליי קבוצת מימים
[NEUTRAL (0)]  (confidence=0.999)  לא ישנתי שנתיים
[DISTRESS (1)]  (confidence=1.000)  השארתי לכם מכתב
[NEUTRAL (0)]  (confidence=1.000)  מה אם לא תזרח השמש מחר אולי יהיה יום אחר
[DISTRESS (1)]  (confidence=1.000)  אני לא יודע מה לעשות
[NEUTRAL (0)]  (confidence=1.000)  קניתי מחשב ואני מתלבט את מי להזמין לשחק אוף
[DISTRESS (1)]  (confidence=0.999)  אף פעם אל תאמר אף פעם


## 11. שמירת המודל והורדה למחשב
לאחר ההורדה, חלצו את הזיפ לתיקיית הפרויקט (למשל `hebert_distress_model/`) ועדכנו ב-`ML.py` את `model_name` לנתיב המקומי הזה.

In [14]:
SAVE_DIR = "hebert_distress_model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import shutil
shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)

from google.colab import files
files.download(f"{SAVE_DIR}.zip")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>